# TM-align 结构比较

使用 TM-align 比较蛋白质结构

## 1. 初始化环境

In [ ]:
import sys
import os
from pathlib import Path

# 添加 protflow 到路径：先尝试从 cwd 向上查找项目根，否则用环境变量 PROTFLOW_ROOT
project_root = Path(os.environ.get('PROTFLOW_ROOT', Path.cwd()))
if not (project_root / 'src' / 'protflow').exists():
    project_root = Path.cwd()
    while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
        project_root = project_root.parent
if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")
else:
    raise SystemError("未找到 ProtFlow 项目根（需包含 src/protflow）。请在项目根目录打开 notebook，或设置环境变量 PROTFLOW_ROOT")

from protflow.utils.notebook_utils import init_notebook, CORE_PACKAGES
paths = init_notebook('tm_align', packages=CORE_PACKAGES)
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

## 2. 导入后端模块

In [ ]:
from protflow.core.structure_comparison import (
    batch_compare_tm_align_from_dir,
    plot_comparison_results,
)

## 3. 运行比较

In [ ]:
structures_dir = DATA_DIR / 'structures'
# 断点续跑：若上次未跑完，设为 True 可跳过已有 (Query,Target) 对并追加写入
RESUME = False

if structures_dir.exists():
    # 从目录扫描 PDB 做两两 TM-align 比对（多进程并行）
    # 高通量时可用: collect_results=False, write_batch_size=50000, num_workers=32
    results = batch_compare_tm_align_from_dir(
        structures_dir=structures_dir,
        output_dir=WORK_DIR / 'comparisons',
        resume=RESUME,
    )
    if results:
        plot_comparison_results(
            results=results,
            output_path=WORK_DIR / 'comparison_plot.png',
        )
    print(f'比较完成: {WORK_DIR}')
else:
    print(f'请设置结构目录: {structures_dir}')

## 4. 按样本目录比对（esm3_structures_by_sample）

若结构按样本分子组织为 `parent_dir/<sample_id>/*.pdb`（如 ESM3 预测输出的 `esm3_structures_by_sample`），请使用专门 notebook：[**23_structure_comparison_tm_align_by_sample.ipynb**](23_structure_comparison_tm_align_by_sample.ipynb)，其调用后端 `run_tm_align_esm3_samples` 完成比对并支持断点续跑。

In [ ]:
# 按样本目录比对请运行：23_structure_comparison_tm_align_by_sample.ipynb